# PCN Visualization Utilities

Interactive Plotly-based plotting functions for visualizing training and evaluation of Predictive Coding Networks.

Functions:
- `plot_energy_history_interactive` — per-batch inference energy trajectories
- `plot_epoch_avg_interactive` — epoch-averaged energy with ±1-std bands
- `plot_train_val_metric` — train/val metric over epochs
- `unflatten_images` — unflatten and tile images for display

## Imports

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly import colors

## Energy History Plots

In [ ]:
def plot_energy_history_interactive(energy_history, title="Batch-averaged Energy Trajectories"):
    """Interactive energy trajectories with Plotly."""
    num_epochs = len(energy_history)
    epoch_colors = colors.sample_colorscale(
        colors.sequential.Viridis,
        [i / (num_epochs - 1) if num_epochs > 1 else 0 for i in range(num_epochs)],
    )

    fig = go.Figure()
    for epoch_idx, epoch_energies in enumerate(energy_history):
        color = epoch_colors[epoch_idx]
        for batch_idx, batch_vals in enumerate(epoch_energies):
            steps = list(range(len(batch_vals)))
            customdata = [[epoch_idx + 1, batch_idx + 1] for _ in steps]
            fig.add_trace(
                go.Scatter(
                    x=steps,
                    y=np.log10(batch_vals),
                    mode="lines",
                    line=dict(color=color, width=1),
                    hovertemplate=(
                        "Epoch %{customdata[0]}<br>"
                        "Batch %{customdata[1]}<br>"
                        "Step %{x}<br>"
                        "Energy %{y:.4f}<extra></extra>"
                    ),
                    customdata=customdata,
                    showlegend=False,
                )
            )

    fig.update_layout(
        title=title, xaxis_title="Inference Step t", yaxis_title="Log10 Energy"
    )
    fig.show()


def plot_epoch_avg_interactive(energy_history, title="Batch-Averaged Energy Trajectories (Mean \u00b11-std)"):
    """Interactive per-epoch mean ±1-std energy trajectories using Plotly."""
    num_epochs = len(energy_history)
    epoch_colors = colors.sample_colorscale(
        colors.sequential.Viridis,
        [i / (num_epochs - 1) if num_epochs > 1 else 0 for i in range(num_epochs)],
        colortype="hex",
    )

    epoch_colors = [
        (
            c
            if isinstance(c, str)
            else "#{0:02x}{1:02x}{2:02x}".format(*(int(round(255 * v)) for v in c))
        )
        for c in epoch_colors
    ]

    fig = go.Figure()

    for epoch_idx, epoch_batches in enumerate(energy_history):
        arr = np.array(epoch_batches)
        mean = arr.mean(axis=0)
        std = arr.std(axis=0)
        steps = list(range(len(mean)))
        color = epoch_colors[epoch_idx]
        fig.add_trace(
            go.Scatter(x=steps, y=mean + std, mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip")
        )
        fig.add_trace(
            go.Scatter(
                x=steps, y=mean - std, mode="lines",
                fill="tonexty",
                fillcolor=f"rgba({int(color[1:3],16)},{int(color[3:5],16)},{int(color[5:7],16)},0.2)",
                line=dict(width=0), showlegend=False, hoverinfo="skip",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=steps, y=mean, mode="lines",
                line=dict(color=color, width=2),
                name=f"Epoch {epoch_idx+1}",
                hovertemplate=(f"Epoch {epoch_idx+1}<br>Step %{{x}}<br>Energy %{{y:.4f}}<extra></extra>"),
            )
        )

    fig.update_layout(title=title, xaxis_title="Inference Step t", yaxis_title="Energy")
    fig.show()

## Train/Val Metric Plot

In [ ]:
def plot_train_val_metric(train_result, val_result, yaxis_title="Metric Name", logy=False):
    """Plot training and validation metrics over epochs using Plotly."""
    train_color = "#440154"
    val_color = "#053061"
    if len(train_result) != len(val_result):
        raise ValueError("train_result and val_result must have the same length.")
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(x=list(range(len(train_result))), y=train_result, mode="lines",
                   line=dict(color=train_color, width=1), name="Train")
    )
    fig.add_trace(
        go.Scatter(x=list(range(len(val_result))), y=val_result, mode="lines",
                   line=dict(color=val_color, width=2), name="Val")
    )
    fig.update_layout(
        title="Training and Validation Metric Over Epochs",
        xaxis_title="Epoch", yaxis_title=yaxis_title,
        legend=dict(title="Split", orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1.0),
    )
    if logy:
        fig.update_yaxes(type="log")
    fig.show()

## Image Utilities

In [ ]:
def unflatten_images(images, image_size=(3, 32, 32), normalize=True):
    """Unflatten images and tile them vertically for display."""
    img = np.asarray(images).reshape(-1, *image_size)
    if normalize:
        img = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8)
    img = (255 * img).astype(np.uint8)
    if image_size[0] == 1:
        img = np.repeat(img, 3, axis=1)
    img = np.concatenate([img[i, :, :, :] for i in range(img.shape[0])], axis=1)
    img = np.transpose(img, (1, 2, 0))
    return img

## Usage Example

After training with `train_pcn(..., return_history=True)`, pass the history to these functions:

```python
plot_energy_history_interactive(energy_history)
plot_epoch_avg_interactive(energy_history)
plot_train_val_metric(train_accs, val_accs, yaxis_title='Accuracy')
```